In [ ]:
import os, sys, subprocess, platform
print(platform.python_version())
subprocess.run(['nvidia-smi'], check=False)
subprocess.run(['bash','-lc','nproc; free -h'], check=False)
import torch
print('torch', torch.__version__, 'CUDA', torch.version.cuda, 'available', torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU runtime')

In [ ]:
RUN_ID = 'rl_seed20261120_run1'
SEED = 20261120
DRIVE_ROOT = '/content/drive/MyDrive/connect4'
MAX_MINUTES = 60
DATASET_NAME = 'solverlabels_20261120_seed20261120_v2'
SOURCE_ZIP = f'{DRIVE_ROOT}/source_bundle_20261120_v5.zip'
INIT_CHECKPOINT = NoneSOURCE_MANIFEST = f'{DRIVE_ROOT}/source_bundle_20261120_v5.manifest.json'


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
root = Path(DRIVE_ROOT); root.mkdir(parents=True, exist_ok=True)
dataset = root/'data'/DATASET_NAME
assert Path(SOURCE_ZIP).is_file(), 'Upload source.zip first'
assert (dataset/'manifest.json').is_file(), 'Upload dataset package first'

In [ ]:
import zipfile, shutil
repo = Path('/content/connect4-project')
if repo.exists(): shutil.rmtree(repo)
repo.mkdir()
import hashlib, json
bundle = json.loads(Path(SOURCE_MANIFEST).read_text())
assert hashlib.sha256(Path(SOURCE_ZIP).read_bytes()).hexdigest() == bundle['zip_sha256'], 'source ZIP checksum mismatch'
with zipfile.ZipFile(SOURCE_ZIP) as z: z.extractall(repo)
candidates = [p for p in repo.iterdir() if p.is_dir()]
if not (repo/'connect4').is_dir() and len(candidates)==1: repo = candidates[0]
def tree_hash(root):
    h=hashlib.sha256()
    for p in sorted(root.rglob('*.py')):
        if any(part in {'.git','.venv','.venv-gpu','__pycache__','.pytest_cache'} for part in p.parts): continue
        h.update(p.relative_to(root).as_posix().encode('utf-8')); h.update(p.read_bytes())
    return h.hexdigest()
assert tree_hash(repo) == bundle['source_tree_sha256'], 'source tree differs from packaged snapshot'


In [ ]:
%cd $repo
!pip install -r requirements-colab.txt

In [ ]:
import hashlib, json
data_manifest = json.loads((dataset/'manifest.json').read_text())
for name, expected in data_manifest['files_sha256'].items():
    assert hashlib.sha256((dataset/name).read_bytes()).hexdigest() == expected, f'checksum mismatch: {name}'
print('frozen test SHA256:', data_manifest['frozen_test_sha256'])

In [ ]:
import os
os.environ['PYTHONPATH'] = str(repo)
out = root/'artifacts'; run_dir = out/RUN_ID
cmd = [sys.executable, '-m', 'connect4.rl_train', '--dataset', str(dataset), '--output-root', str(out), '--run-id', RUN_ID, '--seed', str(SEED), '--device', 'cuda', '--episodes', '10000', '--max-minutes', str(MAX_MINUTES)]
if (run_dir/'checkpoint_latest.pt').is_file(): cmd += ['--resume']
elif INIT_CHECKPOINT: cmd += ['--init-checkpoint', INIT_CHECKPOINT]
print(' '.join(cmd)); subprocess.run(cmd, cwd=repo, check=True)

In [ ]:
import zipfile
archive = root/'deliveries'/f'{RUN_ID}.zip'; archive.parent.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in sorted(run_dir.iterdir()): z.write(p, f'{RUN_ID}/{p.name}')
print('artifact:', archive)